In [1]:
# ============================================================
# PHASE 3 — BASELINE ETA MODEL
# Run: python phase3.py
# ============================================================

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
warnings.filterwarnings('ignore')

# ── Load Phase 2 ─────────────────────────────────────────────
with open('phase2_checkpoint.pkl', 'rb') as f:
    data = pickle.load(f)

df               = data['df']
trip_agg         = data['trip_agg']
corridor_stats   = data['corridor_stats']
hub_df           = data['hub_df']
G                = data['G']
THRESHOLD_SEVERE = data['THRESHOLD_SEVERE']
print(f"✅ Phase 2 loaded | trips: {trip_agg.shape}")

# ── Extended Feature Engineering ─────────────────────────────
trip_agg['osrm_speed']           = (trip_agg['total_osrm_distance'] /
                                    (trip_agg['total_osrm_time']/60).replace(0,np.nan))
trip_agg['distance_per_segment'] = (trip_agg['total_osrm_distance'] /
                                    trip_agg['total_segments'].replace(0,np.nan))
trip_agg['time_per_km_osrm']     = (trip_agg['total_osrm_time'] /
                                    trip_agg['total_osrm_distance'].replace(0,np.nan))
trip_agg['osrm_time_per_segment']= (trip_agg['total_osrm_time'] /
                                    trip_agg['total_segments'].replace(0,np.nan))
trip_agg['severe_x_segments']    = (trip_agg['pct_severe_segments'] *
                                    trip_agg['total_segments'])
trip_agg['is_long_haul']         = (trip_agg['total_osrm_distance'] >
                                    trip_agg['total_osrm_distance'].quantile(0.75)).astype(int)
trip_agg['is_complex_trip']      = (trip_agg['total_segments'] >
                                    trip_agg['total_segments'].quantile(0.75)).astype(int)
trip_agg['log_osrm_distance']    = np.log1p(trip_agg['total_osrm_distance'])
trip_agg['log_osrm_time']        = np.log1p(trip_agg['total_osrm_time'])
trip_agg['log_total_distance']   = np.log1p(trip_agg['total_distance'])

# Frequency encoding
for col, prefix in [('src_state','src_state'),('dst_state','dst_state'),
                    ('src_facility_type','src_ftype'),('dst_facility_type','dst_ftype')]:
    freq = trip_agg[col].value_counts(normalize=True)
    trip_agg[f'{prefix}_freq'] = trip_agg[col].map(freq).fillna(0)

trip_agg['carting_intrastate'] = (
    (trip_agg['route_type_enc']==0) & (trip_agg['is_interstate']==0)
).astype(int)

# ── Feature Sets ─────────────────────────────────────────────
SET_B = [
    'log_osrm_distance','log_osrm_time','log_total_distance',
    'route_type_enc','src_state_freq','dst_state_freq',
    'src_ftype_freq','dst_ftype_freq',
    'creation_hour','creation_dayofweek','creation_month','is_weekend',
    'is_interstate','total_segments','pct_severe_segments',
    'mean_speed_efficiency','osrm_speed','distance_per_segment',
    'time_per_km_osrm','osrm_time_per_segment','severe_x_segments',
    'is_long_haul','is_complex_trip','carting_intrastate',
]

X = trip_agg[SET_B].fillna(0)
y = trip_agg['total_actual_time']

# ── Split ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=trip_agg['route_type']
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# ── Evaluation ───────────────────────────────────────────────
def evaluate(name, y_true, y_pred):
    mae    = mean_absolute_error(y_true, y_pred)
    r2     = r2_score(y_true, y_pred)
    within = np.mean(np.abs(y_true-y_pred)/np.abs(y_true) < 0.15)
    bias   = np.mean(y_pred - y_true)
    print(f"\n{'='*45}\n  {name}\n{'='*45}")
    print(f"  MAE:        {mae:.2f} min")
    print(f"  R²:         {r2:.4f}")
    print(f"  Within 15%: {within:.1%}")
    print(f"  Bias:       {bias:+.2f} min")
    return mae, within

# ── Permutation Importance → Prune ───────────────────────────
print("\nFitting RF for permutation importance...")
rf_b = RandomForestRegressor(n_estimators=300, max_depth=15,
                              min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_b.fit(X_train, y_train)
evaluate("RF Set B", y_test, rf_b.predict(X_test))

print("Computing permutation importance (~2 min)...")
perm = permutation_importance(rf_b, X_test, y_test,
                               n_repeats=10, random_state=42,
                               scoring='neg_mean_absolute_error', n_jobs=-1)
perm_df = pd.DataFrame({
    'feature': SET_B,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

bad_features = perm_df[perm_df['importance'] < 0]['feature'].tolist()
SET_C = [f for f in SET_B if f not in bad_features]
print(f"Pruned: {len(SET_B)} → {len(SET_C)} features | Dropped: {bad_features}")

# ── Retrain on Set C ─────────────────────────────────────────
X_c = trip_agg[SET_C].fillna(0)
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y, test_size=0.2, random_state=42,
    stratify=trip_agg['route_type']
)

# ── Hyperparameter Tuning ─────────────────────────────────────
print("\nRunning RandomizedSearchCV (~5 min)...")
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions={
        'n_estimators':     [200,300,500],
        'max_depth':        [10,15,20,25,None],
        'min_samples_leaf': [3,5,8,10],
        'min_samples_split':[5,10,20],
        'max_features':     ['sqrt',0.4,0.5,0.6],
    },
    n_iter=25, cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42, verbose=1,
    return_train_score=True, n_jobs=-1
)
rf_search.fit(X_c_train, y_c_train)
best_rf = rf_search.best_estimator_

print(f"Best params: {rf_search.best_params_}")
print(f"Best CV MAE: {-rf_search.best_score_:.2f} min")

baseline_mae, baseline_within15 = evaluate(
    "RF Tuned (best params)", y_c_test, best_rf.predict(X_c_test)
)

# ── CV Check ─────────────────────────────────────────────────
cv_scores = cross_val_score(best_rf, X_c_train, y_c_train,
                             cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
cv_mae = -cv_scores
print(f"CV MAE: {cv_mae.mean():.2f} ± {cv_mae.std():.2f} min")

# ── Save Checkpoint ───────────────────────────────────────────
with open('phase3_checkpoint.pkl', 'wb') as f:
    pickle.dump({
        'df':               df,
        'trip_agg':         trip_agg,
        'corridor_stats':   corridor_stats,
        'hub_df':           hub_df,
        'G':                G,
        'THRESHOLD_SEVERE': THRESHOLD_SEVERE,
        'baseline_features':SET_C,
        'y':                y,
        'baseline_mae':     baseline_mae,
        'baseline_within15':baseline_within15,
        'best_rf':          best_rf,
    }, f, protocol=4)

print("\n✅ Phase 3 complete — saved phase3_checkpoint.pkl")

✅ Phase 2 loaded | trips: (14804, 30)
Train: 11843 | Test: 2961

Fitting RF for permutation importance...

  RF Set B
  MAE:        32.08 min
  R²:         0.9801
  Within 15%: 76.9%
  Bias:       -0.75 min
Computing permutation importance (~2 min)...
Pruned: 24 → 22 features | Dropped: ['creation_dayofweek', 'is_long_haul']

Running RandomizedSearchCV (~5 min)...
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best params: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 0.6, 'max_depth': None}
Best CV MAE: 29.64 min

  RF Tuned (best params)
  MAE:        30.95 min
  R²:         0.9816
  Within 15%: 77.6%
  Bias:       -0.78 min
CV MAE: 29.64 ± 0.60 min

✅ Phase 3 complete — saved phase3_checkpoint.pkl
